# DeepSDF generative latent — extra stretch experiments (companion notebook)

Companion to `colab_driver.ipynb`. Run the main notebook's Section 3 (the
scoped N-sweep + D=32 ablation, `OUTPUT_NAME='shapenet_scoped'`) **first** --
this notebook only adds to those existing results on Drive, it does not
reproduce them.

Three things, none of which need the main sweep re-run:

1. **`hidden_dim=512`, post-repair.** Closes an open caveat in the report:
   the D=16→32 ablation reverses post-watertight-fix, but the *original*
   pre-fix "more capacity, worse mesh" symptom was actually about decoder
   width, and width was never re-tested post-repair.
2. **Tighter N=150 generation metrics.** Re-scores the existing `N150_D16`
   checkpoint with `--num-generated`/`--num-reference` raised 40→200 (no
   retraining), removing the $1/40=0.025$ Coverage quantization floor.
3. **Latent-space scatter plot.** A PCA projection of the real codes vs.
   samples from each fitted prior -- the picture behind the Coverage/MMD
   numbers.

**Runtime:** GPU (T4 / L4 / A100), same as the main notebook.

## 1. Setup: clone, install, mount Drive

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/deepsdf-generative-latent')
CODE = REPO / 'code'

if not REPO.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/amitbe711/deepsdf-generative-latent.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull'])

os.chdir(CODE)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
print('Working directory:', os.getcwd())

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/deepsdf_generative')
DRIVE_SHAPENET = Path('/content/drive/MyDrive/shapenet/03001627')
LOCAL_SHAPENET = Path('/content/shapenet/03001627')
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
FIG_ROOT = DRIVE_ROOT / 'figures'

for p in (DRIVE_ROOT, OUTPUT_ROOT, FIG_ROOT):
    p.mkdir(parents=True, exist_ok=True)

os.chdir('/content/deepsdf-generative-latent/code')
print('Drive outputs:', OUTPUT_ROOT)
print('ShapeNet on Drive:', DRIVE_SHAPENET)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. ShapeNet data

Same as the main notebook's Section 2 -- needed here too since the D=512
rerun trains from scratch and the reeval needs the reference-set meshes.
Auto-downloads/copies to `/content/shapenet/03001627` if not already cached
in this runtime.

In [ ]:
import shutil
import zipfile

MESH_LIMIT = 400  # up to N=150 train + up to 200 reference (reeval_grid.py) + margin

DRIVE_ZIP = DRIVE_ROOT / 'data' / '03001627.zip'
LOCAL_ZIP = Path('/content/03001627.zip')


def count_objs(root: Path) -> int:
    return len(list(root.rglob('model_normalized.obj')))


def extract_from_zip(zip_path: Path, dest_dir: Path, limit: int) -> int:
    """Extract first ``limit`` chair models from ShapeNetCore zip."""
    if dest_dir.exists():
        shutil.rmtree(dest_dir)
    dest_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        model_ids: list[str] = []
        for name in zf.namelist():
            if name.endswith('model_normalized.obj'):
                parts = Path(name).parts
                if len(parts) >= 4 and parts[0] == '03001627':
                    model_ids.append(parts[1])
        model_ids = sorted(set(model_ids))[:limit]
        prefixes = tuple(f'03001627/{mid}/' for mid in model_ids)
        for name in zf.namelist():
            if not name.startswith(prefixes) or name.endswith('/'):
                continue
            out = dest_dir.parent / name
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(zf.read(name))
    return count_objs(dest_dir)


def copy_from_dir(src: Path, dest: Path, limit: int) -> int:
    objs = sorted(src.rglob('model_normalized.obj'))[:limit]
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    for obj in objs:
        try:
            rel = obj.relative_to(src)
        except ValueError:
            rel = Path(obj.parent.parent.name) / 'models' / obj.name
        out = dest / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(obj, out)
    return len(objs)


def hf_login() -> None:
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if token:
        login(token=token)
    else:
        print('No HF_TOKEN — trying anonymous download (accept license on HF first)')
        login()


# Re-use meshes already on /content from a previous run, but only if there are
# enough for the *current* MESH_LIMIT.
if count_objs(LOCAL_SHAPENET) >= MESH_LIMIT:
    print(f'Reusing {count_objs(LOCAL_SHAPENET)} meshes at {LOCAL_SHAPENET}')
else:
    src_candidates = [
        DRIVE_SHAPENET,
        Path('/content/drive/MyDrive/shapenet_chairs'),
        Path('/content/drive/MyDrive/ShapeNet/03001627'),
        Path('/content/drive/MyDrive/data/shapenet/03001627'),
    ]
    src = next((p for p in src_candidates if p.exists() and count_objs(p) >= 30), None)

    if src is not None:
        print(f'Copying from Drive: {src}')
        n = copy_from_dir(src, LOCAL_SHAPENET, MESH_LIMIT)
    else:
        zip_path = LOCAL_ZIP
        if DRIVE_ZIP.exists() and not zip_path.exists():
            print('Copying cached zip from Drive...')
            shutil.copy2(DRIVE_ZIP, zip_path)
        if not zip_path.exists():
            from huggingface_hub import hf_hub_download
            hf_login()
            print('Downloading 03001627.zip from Hugging Face (~2 GB, once)...')
            zip_path = Path(
                hf_hub_download(
                    repo_id='ShapeNet/ShapeNetCore',
                    filename='03001627.zip',
                    repo_type='dataset',
                    local_dir='/content/shapenet_dl',
                )
            )
            DRIVE_ZIP.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(zip_path, DRIVE_ZIP)
            print('Cached on Drive:', DRIVE_ZIP)
        print('Extracting mesh subset...')
        n = extract_from_zip(zip_path, LOCAL_SHAPENET, MESH_LIMIT)

    if count_objs(LOCAL_SHAPENET) < min(MESH_LIMIT, 30):
        raise RuntimeError(f'Only {count_objs(LOCAL_SHAPENET)} meshes — check HF access')

!python scripts/prepare_data.py --inspect-dir "{LOCAL_SHAPENET}"
print(f'Ready: {count_objs(LOCAL_SHAPENET)} meshes -> {LOCAL_SHAPENET}')

## 3. Point at the existing main-sweep results

Must match `OUTPUT_NAME` in `colab_driver.ipynb` -- this notebook reads and
extends that run's `summary.json`/checkpoints, it does not create them.

In [ ]:
import yaml

OUTPUT_NAME = 'shapenet_scoped'  # must match colab_driver.ipynb's main sweep
RECON_CAP = 48     # use 32 on T4 if OOM; 48 on A100/L4
T4_SAFE = False    # decode_device=cpu already avoids most OOM; set True only on a free T4

OUTPUT_DIR = OUTPUT_ROOT / OUTPUT_NAME
FIG_DIR = FIG_ROOT / OUTPUT_NAME

if not (OUTPUT_DIR / 'summary.json').exists():
    raise RuntimeError(
        f'{OUTPUT_DIR}/summary.json not found -- run colab_driver.ipynb\'s main '
        'sweep (Section 3) first, this notebook only extends its results.'
    )
print('Output:', OUTPUT_DIR)
print('Figures:', FIG_DIR)

In [ ]:
def patch_config(config_path: Path, mesh_dir: Path, out_yaml: Path, *, t4_safe: bool = True) -> dict:
    cfg = yaml.safe_load(config_path.read_text())
    cfg['data']['source'] = 'mesh_dir'
    cfg['data']['mesh_dir'] = str(mesh_dir)
    cfg['device'] = 'cuda'
    ev = cfg.setdefault('eval', {})
    ev['recon_resolution'] = min(int(ev.get('recon_resolution', 48)), RECON_CAP)
    ev['decode_device'] = 'cpu' if t4_safe else 'cuda'
    if t4_safe:
        ev['max_recon_shapes'] = min(int(ev.get('max_recon_shapes', 10)), 5)
        ev['num_generated'] = min(int(ev.get('num_generated', 20)), 15)
        ev['skip_iou'] = True  # mesh.contains() is RAM-heavy on chairs
    out_yaml.write_text(yaml.dump(cfg))
    print('Patched ->', out_yaml, '(source config:', config_path, ')')
    print('  hidden_dim:', cfg['decoder']['hidden_dim'])
    print('  lr_decoder / lr_codes:', cfg['stage1']['lr_decoder'], '/', cfg['stage1']['lr_codes'])
    print('  recon_resolution:', ev['recon_resolution'])
    print('  decode_device:', ev['decode_device'])
    print('  watertight_pitch:', cfg['data'].get('watertight_pitch', 1.0 / 64.0))
    return cfg

## 4. `hidden_dim=512`, post-repair

Re-runs N=50, D=16 with decoder `hidden_dim` 256→512 (lr dropped 1e-3→3e-4, a
separately-diagnosed optimizer-stability fix, not part of the capacity
variable under test) and everything else identical to the main sweep's
`N50_D16` cell. Writes to its own output dir, cannot clobber `OUTPUT_DIR`.

In [ ]:
D512_CONFIG = 'configs/shapenet_n50_D512.yaml'
D512_NAME = 'shapenet_n50_D512'

D512_DIR = OUTPUT_ROOT / D512_NAME
D512_YAML = Path('/content/run_config_d512.yaml')
D512_DIR.mkdir(parents=True, exist_ok=True)

patch_config(CODE / D512_CONFIG, LOCAL_SHAPENET, D512_YAML, t4_safe=T4_SAFE)

print('Config:', D512_CONFIG)
print('Output:', D512_DIR)

In [ ]:
# Single cell, hidden_dim=512 instead of 256, otherwise identical to the main
# sweep's N50_D16 cell. Resumable: re-running this cell after a disconnect
# skips straight past it if metrics.json already exists.
cmd = f'python scripts/run_grid.py --config {D512_YAML} --output {D512_DIR}'
print(cmd)
!{cmd}

In [ ]:
import json

main_summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
d512 = json.loads((D512_DIR / 'N50_D512' / 'metrics.json').read_text())

print(f"{'':16s}{'chamfer':>10s}{'iou':>10s}")
for cell in main_summary:
    if cell['N'] == 50:
        r = cell['reconstruction']
        print(f"  D={cell['D']:<4d} (main){r['chamfer']:>10.4f}{r['iou']:>10.3f}")
r = d512['reconstruction']
print(f"  D=512 (postfix){r['chamfer']:>10.4f}{r['iou']:>10.3f}")
print()
print('Pre-fix reference: D=16->32 made chamfer worse (0.129 -> 0.540) before the')
print('watertight repair. If D=512 also now helps or is at least not worse -- like')
print('the post-fix D=32 ablation -- that is a second, independent confirmation')
print('that capacity behaves normally once the SDF labels are fixed.')

## 5. Tighter N=150 generation metrics

Re-scores the existing `N150_D16` checkpoint at `--num-generated 200
--num-reference 200` (up from 40), no retraining. Removes the $1/40=0.025$
Coverage quantization floor and reduces 1-NN's small-sample bias toward
$1.0$.

In [ ]:
# Re-scores the existing N150_D16 checkpoint only -- no retraining. Resumable
# and idempotent: a cell already scored at these exact eval settings is
# skipped unless --force is passed. The previous metrics.json is preserved as
# metrics_pre_reeval.json (first run only), so the original numbers stay
# recoverable for the comparison in the next cell.
cmd = f'python scripts/reeval_grid.py --output {OUTPUT_DIR} --only-N 150 --only-D 16 --num-generated 200 --num-reference 200'
print(cmd)
!{cmd}

In [ ]:
import json

metrics_path = OUTPUT_DIR / 'N150_D16' / 'metrics.json'
backup_path = metrics_path.with_name('metrics_pre_reeval.json')
current = json.loads(metrics_path.read_text())

print(f"{'':16s}{'coverage':>10s}{'mmd':>10s}{'1-nn':>10s}{'valid':>10s}")
if backup_path.exists():
    before = json.loads(backup_path.read_text())
    for name, g in before['generators'].items():
        print(f"  {name:<8s}(40) {g['coverage']:>9.3f}{g['mmd']:>10.4f}{g['one_nn_acc']:>10.3f}{g['valid_ratio']:>10.2f}")
else:
    print('  (no metrics_pre_reeval.json -- cell above may already have run before)')
for name, g in current['generators'].items():
    print(f"  {name:<8s}(200){g['coverage']:>9.3f}{g['mmd']:>10.4f}{g['one_nn_acc']:>10.3f}{g['valid_ratio']:>10.2f}")
print()
print('summary.json for the whole grid was rewritten in place -- the figures')
print('cell below will pick up these new N150_D16 numbers automatically.')

## 6. Regenerate figures (incl. the new latent-space scatter)

Rewrites `results_table.tex` and all PNGs in `FIG_DIR` from the current
`summary.json` -- picks up the reeval'd N=150 numbers from Section 5
automatically. Copy the updated files into `report/figures/`.

In [ ]:
!python scripts/make_figures.py --input "{OUTPUT_DIR}" --figures "{FIG_DIR}" --latent-scatter-samples 200

from IPython.display import Image, display
for name in ['degradation_generation.png', 'degradation_reconstruction.png', 'loss_curves.png', 'gallery.png', 'latent_scatter.png']:
    p = FIG_DIR / name
    if p.exists():
        display(Image(filename=str(p)))